# Automatization extraction from docstrings


In [41]:
import fine as fn  # Provides objects and functions to model an energy system
from IPython.display import display
import inspect
import pandas as pd
from fine.component import *
from fine.storage import *
import re
pd.set_option("display.max_colwidth", None)

def init_to_table_strong(cls): # Function that extracts parameter information from a class constructor (storage, component etc...)
    doc = inspect.getdoc(cls.__init__) or "" # Retrieve the __init__ docstring.

    rows = []

    param_pattern = re.compile(r":param\s+([A-Za-z0-9_]+)\s*:\s*(.*)") # Regex pattern to extract parameter names, descriptions and type  from Sphinx-style docstrings:
    type_pattern = re.compile(r":type\s+([A-Za-z0-9_]+)\s*:\s*(.*)")

    param_data = {} # Dictionary used to store extracted information for each parameter.

    # Iterate through all lines of the docstring.
    
    for line in doc.split("\n"):
        line = line.strip()

        # extract parameter name and description
        p = param_pattern.match(line)
        if p:
            name = p.group(1)
            desc = p.group(2).strip()

            param_data.setdefault(name, {})
            param_data[name]["description"] = desc
            continue

        # extract "type" value, if its empty return "see below"

        t = type_pattern.match(line)
        if t:
            name = t.group(1)
            typ = t.group(2).strip()

            param_data.setdefault(name, {})

            if not typ:
                typ = "see below"

            param_data[name]["type"] = typ
            continue

        # extract "default" value
        
        if "default value" in line.lower():
            for name in param_data:
                if name.lower() in line.lower():
                    param_data[name]["default"] = line.split("default value is")[-1].strip()

    sig = inspect.signature(cls.__init__)

    for name, param in sig.parameters.items():
        if name == "self":
            continue

        rows.append({
            "Argument": name,
            "Description": param_data.get(name, {}).get("description", ""),
            "Type": param_data.get(name, {}).get("type", ""),   # ✅ FIX ICI
            "Default": param_data.get(name, {}).get(
                "default",
                param.default if param.default != inspect._empty else "/"
            ),
        })

    return pd.DataFrame(rows)


df = init_to_table_strong(Storage)
display(df)


,Argument,Description,Type,Default
0,esM,,,/
1,name,,,/
2,commodity,to the component related commodity.,string,/
3,chargeRate,ratio of the maximum storage inflow (in commodityUnit/hour) to the,0 < float,1
4,dischargeRate,ratio of the maximum storage outflow (in commodityUnit/hour) to,0 < float,1
5,chargeEfficiency,defines the efficiency with which the storage can be charged (equals,0 <= float <=1,1
6,dischargeEfficiency,defines the efficiency with which the storage can be discharged,0 <= float <=1,1
7,selfDischarge,percentage of self-discharge from the storage during one hour,0 <= float <=1,0
8,cyclicLifetime,"if specified, the total number of full cycle equivalents that are supported",None or positive float,None
9,stateOfChargeMin,threshold (percentage) that the state of charge can not drop under,,0


### Initialize the Energy System Model

The Energy System Model is initialized using the `EnergySystemModel` class.
The following table shows all arguments and their default values. Required parameters are in *italic*.

| Argument | Description | Default (if not required)| More Information |
| --- | --- | --- | --- |
| *locations* | set of strings representing the regions in the energy system | / | see below |
| *commodities* | set of strings where each string represents a commodity | / | see below |
| *commoditiyUnitsDict* | dictionary that maps every commodity to a quantitative unit per time | / | see below |
| numberOfTimeSteps | defines the number of time steps considered |  8760 | see below |
| hoursPerTimeStep | defines how many hours are represented by a single time step | 1 | see below |
| startYear |defines the year of the first investment period | 0 | see below |
| numberOfInvestmentPeriods | MISSING | 1 | see below |
| investmentPeriodInterval | defines the time distance between investment period | 1 | see below |
| stochasticModel | determines whether the optimization problem is solved as a stochastic optimization | 1 | [Example](../../09_Stochastic_Optimization\09_Stochastic%20Optimization.ipynb) |
| costUnit | defines the unit used for all cost values | 1e9 Euro | see below |
| lengthUnit | defines the unit used for all length-related values | km | see below |
| verboseLogLevel | controls the amount of logging printed to the console | 0 | see below |
| balanceLimit | defines constraints that limit commodity balances in the system | None | see Example (LINK MISSING) |
| pathwayBalanceLimit | defines commodity balance limits for the entire transformation pathway | None | see Example (LINK MISSING) |
| annuityPerpetuity | determines whether the model assumes that the design of the last investment period continues indefinitely | False | see below |



## Required Arguments

### locations

`locations` is a **set of strings**, where each string represents a region or node in the modeled energy system. Components such as sources, sinks, or converters are always assigned to one or more of these locations.

Examples:
- Countries (e.g. "Germany", "France")
- Regions within a country
- Individual sites such as industrial plants or cities

In this example, we consider two regions:

### commodities

`commodities` defines the **energy carriers** or **materials** that are considered in the energy system model.

It is a **set of strings**, where each string represents a commodity that can be produced, consumed, converted, or transported within the model.

Examples:
- electricity
- hydrogen
- natural gas
- heat
- CO<sub>2</sub>

In this example, we consider the following commodities:

### commodityUnitsDict


`commodityUnitsDict` defines the **unit used for each commodity**.

It is a dictionary that maps every commodity to a **quantitative unit per time**. These units are mainly used for interpreting and presenting results.

Examples:
- GW<sub>el</sub>
- GW<sub>H<sub>2</sub></sub>
- GW<sub>CH<sub>4</sub>, LHV </sub>
- Mio. t<sub>CO<sub>2</sub></sub>/h

Choosing appropriate units is important for the **interpretability of results**. For example, electricity might be expressed in GW<sub>el</sub>, while hydrogen flows might be given in t/h.

The three commodities included in this example will be assigned to their units as follows:

## Default Arguments


The following arguments have **default values** and are optional.

### numberOfTimeSteps

`numberOfTimeSteps` defines the **number of time steps considered when modeling the energy system**.

For each time step, the model creates variables and constraints describing the operation of the system. Together with the `hoursPerTimeStep`, the total number of hours considered can be derived. The total number of hours is again used for scaling the arising costs to the arising **total annual costs (TAC)**, which are minimized during optimization.

Type: strictly positive integer<br>
Default value: 8760 (corresponds to **one time step per hour for a full year**)

### hoursPerTimeStep

`hoursPerTimeStep` defines how many **hours are represented by a single time step**.

Type: strictly positive float<br>
Default value: 1

### numberOfInvestmentPeriods

MISSING

### investmentPeriodInterval

`investmentPeriodInterval` defines the **time distance between investment period**.

This parameter is relevant when performing **multi-period planning** or **transformation pathway analyses**.

Example:<br>
If the investment periods are 2020, 2025 and 2030, the `investmentPeriodInterval` is 5.

Type: strictly positive integer<br>
Default value: 1

### startYear

`startYear` defines the **year of the first investment period**.

Example:<br>
If the investment periods are 2020, 2025 and 2030, the `startYear` is 5.

Type: integer<br>
Default value: 0

### stochasticModel

`stochasticModel` determines whether the optimization problem is solved as a **stochastic optimization**.

In stochastic optimization, the model considers multiple possible future conditions and finds a system design that performs well under all of them.

Examples of uncertainties that could be modeled include:
- different weather years
- varying demand scenarios
- uncertain fuel prices

In this case, the investment periods represent **different scenarios** rather than a transformation pathway.

Type: boolean<br>
Default value: False

### costUnit

`costUnit` defines the **unit used for all cost values** in the energy system model.

Examples:
- 10^9 Euro (billion euros), which can be a suitable scale for national energy systems.
- 1e6 Euro (million euros)

Type: string<br>
Default value: "10^9 Euro"

### lengthUnit

`lengthUnit` defines the unit used for **all length-related values** in the energy system.

This is relevant for components such as **transmission infrastructure**.

Type: string
Default value: 'km'

### verboseLogLevel

`verboseLogLevel` controls the **amount of logging printed to the console**.

Available options:
| Value | Behavior |
| --- | --- |
| 0 | General model logging, warnings and optimization solver logging are displayed |
| 1 | Only warnings are displayed |
| 2 | No general model logging or warnings are displayed, optimization solver logging is set to a minimum |

If required, solver logging can also be configured separately during the optimization step.

### balanceLimit

`balanceLimit` defines constraints that limit **commodity balances** in the system such as production, consumption, imports, or emissions.

Examples:
- CO<sub>2</sub> emission limits
- Minimal renewable generation
- Maximum fuel extraction

Because this concept is more complex, it is explained in detail on a separate documentation page.

Type:
- pd.DataFrame
- dictionary with investment periods as keys, and pd.DataFrame as values

Default value: None

### pathwayBalanceLimit

`pathwayBalanceLimit` defines **commodity balance limits for the entire transformation pathway**.

Unlike `balanceLimit`, this constraint applies to the **whole pathway instead of individual investment periods**.

Example:
- total CO<sub>2</sub> budget across the entire transformation pathway

Type: pd.DataFrame <br>
Default value: None

### annuityPerpetuity

`annuityPerpetuity` determines whether the model assumes that the design of the last investment period continues indefinitely.

If enabled:
- the system design of the final investment period is assumed to remain unchanged forever
- costs are adjusted using the component's interest rate to represent perpetuity costs

To use this feature, **all components must have an interest rate greater than zero**.

Type: boolean<br>
Default value: False